# ???

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.optim as optim
import warnings
warnings.filterwarnings("ignore")
%matplotlib inline

# ??????????

In [ ]:
features = pd.read_excel('D:\\typhoon(new).xlsx')

features.head()

## ????
#### SID:?????????
#### SEASON:?????
#### NUMBER:????????
#### BASIN:?????????
#### SUBBASIN:?????????
#### NAME:????
#### ISO_TIME:????
#### NATURE:????
#### TRACK_TYPE:??????
#### LAT:??
#### LON:??
#### DIST2LAND:??????????
#### STORM_SPEED:????
#### STORM_DIR:????
#### USA_WIND:???????????
#### USA_SSHS:????

In [ ]:
print('????:',features.shape)

# ???

#### ?4??????

In [ ]:
# ???????????"Year"
features['Year'] = pd.to_datetime(features['ISO_TIME']).dt.year

# ?"SID"?????,???????,???????????
features_unique = features.drop_duplicates(subset='SID')

# ?????????????
features_unique['Year_Group'] = features_unique['Year'].apply(lambda x: (x // 5) * 5)

# ?????????????
yearly_counts = features_unique['Year_Group'].value_counts().sort_index()

yearly_counts_df = pd.DataFrame({'Year Group': yearly_counts.index, 'Number of Typhoons': yearly_counts.values})
yearly_counts_df.to_csv(r'D:\five_counts.csv', index=False)

# ??????????????
plt.bar(yearly_counts.index, yearly_counts.values)
plt.xlabel('Year Group')
plt.ylabel('Number of Typhoons')
plt.title('Number of Typhoons per Five Year')
plt.xticks(yearly_counts.index)
plt.show()


#### ??????????

In [ ]:
# ????NATURE?????
nature_counts = features['NATURE'].value_counts()

nature_counts_df = pd.DataFrame({'Nature': nature_counts.index, 'Count': nature_counts.values})
nature_counts_df.to_csv(r'D:\nature.csv', index=False)

# ?????????NATURE?????
plt.bar(range(len(nature_counts)), nature_counts.values)
plt.xlabel('Nature')
plt.ylabel('Number of Typhoons')
plt.title('Number of Typhoons by Nature')
plt.xticks(range(len(nature_counts)), nature_counts.index, rotation=45)
plt.show()

#### ???????????

In [ ]:
# ????NATURE?????
basin_counts = features['BASIN'].value_counts()

basin_counts_df = pd.DataFrame({'Basin': basin_counts.index, 'Count': basin_counts.values})
basin_counts_df.to_csv(r'D:\basin.csv', index=False)

# ?????????NATURE?????
plt.bar(range(len(basin_counts)), basin_counts.values)
plt.xlabel('BASIN')
plt.ylabel('Number of Typhoons')
plt.title('Number of Typhoons by BASIN')
plt.xticks(range(len(basin_counts)), basin_counts.index, rotation=45)
plt.show()

# ??LSTM??

## ???????

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras import backend as K
from tensorflow.keras.layers import Reshape
from tensorflow.keras.layers import TimeDistributed
from tensorflow.keras.layers import RepeatVector
import tensorflow as tf
import joblib

# ????????
selected_features = ['BASIN', 'NATURE', 'TRACK_TYPE', 'LAT', 'LON', 'DIST2LAND', 'STORM_SPEED', 'STORM_DIR', 'USA_WIND', 'USA_SSHS']
features_selected = features[selected_features]
#features_selected.head()

# ????
features_encoded = pd.get_dummies(features_selected, columns=['BASIN', 'NATURE', 'TRACK_TYPE'])
#features_encoded.head()
#print(features_encoded.dtypes)

X = features_encoded.values
#y = features_selected[['LAT', 'LON', 'USA_SSHS']].values.astype(float)
#y = features_selected[['LAT', 'LON']].values.astype(float)
y = features_selected[['LAT', 'LON','USA_WIND','DIST2LAND']].values.astype(float)

# ????????????????
timesteps = 1
#timesteps = 5  # ??????,??????????
X_sequence = []
y_sequence = []
#print(X[5])

n = len(X)
for i in range(timesteps, n-5):
    X_sequence.append(X[i-timesteps:i])
    y_sequence.append(y[i:i+5])
    
#print(X_sequence[5])
X_sequence = np.array(X_sequence).astype(np.float32)
y_sequence = np.array(y_sequence).astype(np.float32)

# ???X_sequence
scaler_X = StandardScaler()
X_sequence_scaled = scaler_X.fit_transform(X_sequence.reshape(-1, X_sequence.shape[-1])).reshape(X_sequence.shape)
#print(X_sequence_scaled[5])

# ???y_sequence
scaler_y = StandardScaler()
y_sequence_scaled = scaler_y.fit_transform(y_sequence.reshape(-1, y_sequence.shape[-1])).reshape(y_sequence.shape)
#print(y_sequence_scaled[5])

# ???????
joblib.dump(scaler_X, 'scaler_X.pkl')
joblib.dump(scaler_y, 'scaler_y.pkl')

# ??LSTM??
model = Sequential()
model.add(LSTM(64, input_shape=(timesteps, X_sequence.shape[2])))
model.add(RepeatVector(5))  # ????5?,??????(?, 5, 64)
model.add(LSTM(64, return_sequences=True))  # ?????????(?, 5, 64)
model.add(TimeDistributed(Dense(3)))  # ???????????(?, 5, 3)
#model.add(TimeDistributed(Dense(2)))

# ????
model.compile(optimizer='adam', loss='mse')

# ????
model.fit(X_sequence_scaled, y_sequence_scaled, epochs=10, batch_size=32)

# ????
model.save('lstm_model.h5')

## ????

In [ ]:
from tensorflow.keras.models import load_model
from sklearn.model_selection import train_test_split

# ????????
model = load_model('lstm_model.h5')

# ?????????
X_train, X_val, y_train, y_val = train_test_split(X_sequence_scaled, y_sequence_scaled, test_size=0.00005, random_state=42)

# ???????????
val_predictions = model.predict(X_val)
val_predictions = scaler_y.inverse_transform(val_predictions.reshape(-1, y_sequence.shape[-1])).reshape(val_predictions.shape)

#???
timesteps = X_val.shape[1]
val_steps = y_val.shape[1]

# ???????????
y_val = scaler_y.inverse_transform(y_val.reshape(-1, y_sequence.shape[-1])).reshape(y_val.shape)

print(y_val)

# # ????????????
# fig, axs = plt.subplots(val_steps, 2, figsize=(10, 5*val_steps))

# for i in range(val_steps):
#     axs[i, 0].plot(y_val[:, i, 0], label='True')
#     axs[i, 0].plot(val_predictions[:, i, 0], label='Predicted')
#     axs[i, 0].set_title('Latitude - Step {}'.format(i+1))
#     axs[i, 0].legend()
    
#     axs[i, 1].plot(y_val[:, i, 1], label='True')
#     axs[i, 1].plot(val_predictions[:, i, 1], label='Predicted')
#     axs[i, 1].set_title('Longitude - Step {}'.format(i+1))
#     axs[i, 1].legend()

# plt.tight_layout()
# plt.show()

## ????

In [ ]:
# ????????
model = tf.keras.models.load_model('lstm_model.h5')

# ????
input_data = pd.DataFrame({
    'LAT': [-12.5],
    'LON': [172.5],
    'DIST2LAND': [647],
    'STORM_SPEED': [6],
    'STORM_DIR': [351],
    'USA_WIND': [25],
    'USA_SSHS': [-1],
    'BASIN_EP':[0],
    'BASIN_NI':[0],
    'BASIN_SA':[0],
    'BASIN_SI':[0],
    'BASIN_SP':[1],
    'BASIN_WP':[0],
    'NATURE_DS':[0],
    'NATURE_ET':[0],
    'NATURE_MX':[0],
    'NATURE_NR':[0],
    'NATURE_SS':[0],
    'NATURE_TS':[1],
    'TRACK_TYPE_PROVISIONAL':[0],
    'TRACK_TYPE_PROVISIONAL_spur':[0],
    'TRACK_TYPE_main':[1],
    'TRACK_TYPE_spur-merge':[0],
    'TRACK_TYPE_spur-other':[0],
    'TRACK_TYPE_spur-split':[0]
})

# ???????
input_data_scaled = scaler_X.transform(input_data)

# ?????????????
input_sequence = np.array([input_data_scaled])  # ??:(1, timesteps, num_features)

# ????
predicted_sequence_scaled = model.predict(input_sequence)  # ??:(1, 5, num_features)
predicted_sequence_2d = np.reshape(predicted_sequence_scaled, (5, -1))  # ????????? (5, num_features)
predicted_sequence = scaler_y.inverse_transform(predicted_sequence_2d)  # ????

# ?????

y1 = predicted_sequence[0]
y2 = predicted_sequence[1]
y3 = predicted_sequence[2]
y4 = predicted_sequence[3]
y5 = predicted_sequence[4]

# ????
print("?????????,????????:", y1, y2, y3, y4, y5)

# # ?????
# lat_predictions = predicted_sequence[:, 0]  # ?????????(LAT)
# lon_predictions = predicted_sequence[:, 1]  # ?????????(LON)
# # ????
# plt.plot(lat_predictions, lon_predictions, marker='o')



# # ?????
# plt.scatter(input_data['LAT'], input_data['LON'], c='blue', label='Input', zorder=10)
# plt.plot([input_data['LAT'], y1[0]], [input_data['LON'], y1[1]], 'p-', label='Line')


# plt.xlabel('LAT')
# plt.ylabel('LON')
# plt.legend()

# plt.show()